# Build the resource library (Firestore)

Embeds every entry in `data/resources.json` (title + summary) with Gemini and upserts it into a Firestore collection as a document with a `Vector` field, so it can be searched with Firestore's native `find_nearest()` vector query — a real managed database instead of a local file.

**One-time setup** (see `../README.md`):
1. Enable the Firestore API and create a Native-mode database in your GCP project.
2. Create a vector index on the `embedding` field (requires the `gcloud` CLI):
```
gcloud firestore indexes composite create \
  --collection-group=resources --query-scope=COLLECTION \
  --field-config field-path=embedding,vector-config='{"dimension":"768","flat":"{}"}'
```
3. `pip install google-cloud-firestore`, with `.env` configured and `gcloud auth application-default login` already run.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'samples')))


## Step 1 — load the curated resource list

`data/resources.json` holds real, verified links (papers, docs, courses) across agents, agent toolkits, LLM fundamentals, RAG, embeddings, and search.

In [ ]:
from resource_library import load_resources

resources = load_resources()
print(f'{len(resources)} resources')
for r in resources[:5]:
    print(f"  [{r['topic']}] {r['title']}")


## Step 2 — embed each resource

Embed `"{title}. {summary}"` for every entry with `task_type="RETRIEVAL_DOCUMENT"` — the document side of Gemini's retrieval-optimized embedding space.

In [ ]:
from resource_library import embed_texts

vectors = embed_texts(
    [f"{r['title']}. {r['summary']}" for r in resources],
    task_type='RETRIEVAL_DOCUMENT',
)
print(len(vectors), 'vectors, dimension', len(vectors[0]))


## Step 3 — upsert into Firestore

Each resource becomes one Firestore document (id = the resource's `id` field) with an `embedding` field holding a `Vector`. A `batch()` write commits all of them in one round trip.

In [ ]:
from google.cloud.firestore_v1.vector import Vector
from resource_library import get_firestore_client, COLLECTION

db = get_firestore_client()
batch = db.batch()
for resource, vector in zip(resources, vectors):
    doc_ref = db.collection(COLLECTION).document(resource['id'])
    batch.set(doc_ref, {**resource, 'embedding': Vector(vector)})
batch.commit()
print(f'Upserted {len(resources)} resources into \'{COLLECTION}\'')


## Step 4 — query it

`search()` embeds the question with `task_type="RETRIEVAL_QUERY"` and runs Firestore's `find_nearest()` server-side — no local vector math.

In [ ]:
from resource_library import search

for hit in search('how do I learn to build a RAG agent?'):
    print(f"[{hit['distance']:.3f}] {hit['title']} — {hit['url']}")
